In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Load Data & Preprocessing
df = (
    pd.read_csv('supply_chain_data_part3_4.csv')
    .assign(
        timestamp=lambda x: pd.to_datetime(x['timestamp'], errors='coerce'),
        day_of_week=lambda x: pd.to_datetime(x['timestamp'], errors='coerce').dt.day_name(),
        driver_risk_score=lambda x: (1 - x['driver_behavior_score']) * x['fatigue_monitoring_score']
    )
    ## Dropna from those columns
    .dropna(subset=['iot_temperature', 'driver_behavior_score', 'fuel_consumption_rate'])
)

    ## Classifying features & Train/Test split
target = 'delivery_time_deviation'
raw_num_features = [
    'traffic_congestion_level', 'warehouse_inventory_level', 'loading_unloading_time',
    'handling_equipment_availability', 'weather_condition_severity', 'port_congestion_level',
    'shipping_costs', 'supplier_reliability_score', 'lead_time_days', 'historical_demand',
    'route_risk_level', 'driver_behavior_score', 'fatigue_monitoring_score'
]
num_features = raw_num_features + ['driver_risk_score']
cat_features = ['day_of_week']

    ## Drop N/A from modeling data
df = df.dropna(subset=num_features + cat_features + [target])

X_train, X_test, y_train, y_test = train_test_split(
    df[num_features + cat_features], df[target], test_size=0.2, random_state=42
)



# 2. Preprocessing pipeline
tree_preprocessor = ColumnTransformer(
    [('cat', OneHotEncoder(drop='first'), cat_features)],
    remainder='passthrough'
)

linear_preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(drop='first'), cat_features)
])



# 3. Set model & hyperparemeter
models = {
    "RandomForest": (
        tree_preprocessor,
        RandomForestRegressor(random_state=42, n_jobs=-1),
        {   # 2×2×2×2 = 16
            'model__n_estimators': [300, 600],
            'model__max_depth': [None, 12],
            'model__min_samples_leaf': [1, 5],
            'model__max_features': ['sqrt', 'log2']
        }
    ),
    "GradientBoosting": (
        tree_preprocessor,
        GradientBoostingRegressor(random_state=42),
        {   # 2×2×2 = 8
            'model__n_estimators': [300, 600],
            'model__learning_rate': [0.05, 0.1],
            'model__max_depth': [2, 3]
        }
    ),
    "Ridge": (
        linear_preprocessor,
        Ridge(random_state=42),
        {'model__alpha': [0.1, 1.0, 10.0]}
    ),
    "Lasso": (
        linear_preprocessor,
        Lasso(random_state=42),
        {'model__alpha': [0.001, 0.01, 0.1]}
    ),
    "ElasticNet": (
        linear_preprocessor,
        ElasticNet(random_state=42),
        {'model__alpha': [0.01, 0.1, 1.0], 'model__l1_ratio': [0.2, 0.5, 0.8]}
    ),
    "LinearRegression": (
        linear_preprocessor,
        LinearRegression(),
        {}
    ),
}



# 4. Train & Test loop
results = []
best_models = {}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, (preprocessor, estimator, params) in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    gs = GridSearchCV(pipe, params, scoring='neg_root_mean_squared_error', cv=cv, n_jobs=-1)
    gs.fit(X_train, y_train)

    preds = gs.predict(X_test)
    results.append({
        "Model": name,
        "CV_RMSE": -gs.best_score_,
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "Test_MAE": mean_absolute_error(y_test, preds),
        "Test_R2": r2_score(y_test, preds),
        "Best_Params": gs.best_params_
    })
    best_models[name] = gs

    ## Results 
results_df = pd.DataFrame(results).sort_values("Test_RMSE").reset_index(drop=True)
baseline_rmse = np.sqrt(mean_squared_error(y_test, np.full(len(y_test), y_train.mean())))

print(f"Baseline RMSE: {baseline_rmse:.4f}")
display(results_df)


Baseline RMSE: 4.4965
Data: 30,204 rows → Train 24,163 / Test 6,041



,Model,CV_RMSE,Test_RMSE,Test_MAE,Test_R2,Best_Params
0,LinearRegression,3.413776,3.355165,2.696860,0.443231,{}
1,Ridge,3.413719,3.355236,2.696936,0.443207,{'model__alpha': 10.0}
2,ElasticNet,3.413234,3.356426,2.698627,0.442812,"{'model__alpha': 0.01, 'model__l1_ratio': 0.8}"
3,Lasso,3.413214,3.356522,2.698772,0.442780,{'model__alpha': 0.01}
4,GradientBoosting,3.426888,3.364857,2.711241,0.440010,"{'model__learning_rate': 0.05, 'model__max_dep..."
5,RandomForest,3.448577,3.382660,2.733422,0.434068,"{'model__max_depth': 12, 'model__max_features'..."
